# 🛒 고객 문의 유형 분류기 (v3 — 모델 & 카테고리 개선)
**유형:** 분류형 (임베딩 + SVM/로지스틱 회귀 비교)  
**목표:** 고객 문의 내용을 입력하면 어떤 유형(AS, 결제, 교환, 반품, 배송, 주문)인지 예측하는 AI 만들기

---
### 데이터 출처
- **K쇼핑 콜센터 질의응답 데이터** (AI Hub 민원 콜센터 데이터셋)
- 6개 카테고리: AS, 결제, 교환, 반품, 배송, 주문
- 고객 질문 + **상담원 응답** + **고객 의도** 결합하여 분류기 학습

### 버전별 개선 히스토리
| 개선 항목 | v1 | v2 | v3 |
|-----------|----|----|-----|
| 임베딩 모델 | MiniLM (384차원) | MiniLM (384차원) | **MPNet (768차원)** |
| 카테고리 | 7개 | 7개 | **6개 (업무처리 제거)** |
| 입력 텍스트 | 고객 발화만 | 고객+상담원+의도 | 고객+상담원+의도 |
| 학습 데이터 | 500건/카테고리 | 1000건/카테고리 | 1000건/카테고리 |
| 분류기 | 로지스틱 회귀 | LR vs SVM 비교 | LR vs SVM 비교 |
| 평가 | 분류 리포트 | +혼동 행렬 | +혼동 행렬 |

### v3 핵심 개선 사항
1. **임베딩 모델 업그레이드**: `MiniLM-L12` (384차원) → `MPNet-base` (768차원)
   - 2배 넓은 벡터 공간에서 더 세밀한 의미 구분 가능
2. **"업무처리" 카테고리 제거**
   - 데이터 분석 결과, 업무처리의 top 의도가 "방송상품주문", "상품주문신청" 등으로 **주문 카테고리와 거의 동일**
   - 독립적 의미가 없는 catch-all 카테고리여서 다른 카테고리와 심하게 혼동됨 (v2에서 43% 정확도)
   - 제거함으로써 나머지 6개 카테고리의 경계가 더 명확해짐

### 이 노트북의 구조
| 단계 | 내용 |
|------|------|
| Step 1 | 데이터 로드 (고객+상담원+의도 결합) |
| Step 2 | 데이터 전처리 |
| Step 3 | 임베딩 & 분류기 학습 (모델 비교) |
| Step 4 | 예측 & 평가 |
| Step 5 | 미니 미션 |

### 왜 데이터 전처리가 중요한가?
원본 데이터는 **콜센터 통화 녹취를 텍스트로 변환**한 것이라 다음과 같은 문제가 있습니다:
- `"네."`, `"아 그래요?"` 같은 **의미 없는 짧은 응답**이 전체의 3~5%
- 한 통화의 고객 발화가 **문장 단위로 쪼개져** 있어 맥락 없는 단편만 존재
- `"ㅇㅇㅇ"` 같은 **개인정보 마스킹 토큰**이 포함
- `"아 그래요?"`가 **모든 카테고리에 공통 등장** → 노이즈

**전처리 없이는 AI가 노이즈를 학습하게 됩니다!**

## Step 1: 데이터 로드 (원본 그대로)

In [ ]:
# =============================================================================
# 1. 필수 라이브러리 설치 및 임포트
# =============================================================================
# [의도] 이 프로젝트에서 사용하는 모든 외부 라이브러리를 설치하고 불러옵니다.
#   - sentence-transformers: 텍스트를 고차원 벡터(임베딩)로 변환하는 사전학습 모델 라이브러리
#   - scikit-learn: 분류기(LogisticRegression, SVM) 학습 및 성능 평가 도구
#   - torch: sentence-transformers의 내부 연산을 위한 딥러닝 프레임워크(PyTorch)
#   - python-dotenv: .env 파일에서 환경변수를 읽어오는 라이브러리
# =============================================================================
!pip install sentence-transformers scikit-learn torch python-dotenv

# json: K쇼핑 콜센터 데이터가 JSON 형식으로 저장되어 있으므로 파싱에 필요
# zipfile: 데이터가 zip 압축 파일로 제공되므로 압축 해제에 필요
# re: 텍스트 전처리 시 정규표현식으로 노이즈를 제거하기 위해 사용
# os: 환경변수(데이터 경로)를 읽어오기 위해 사용
import json, zipfile, re, os

# pandas: 표 형태의 데이터(DataFrame)를 다루기 위한 핵심 라이브러리
# numpy: 임베딩 벡터 등 수치 배열 연산에 사용
import pandas as pd
import numpy as np

# Path: 운영체제에 독립적인 파일 경로 처리를 위해 사용
from pathlib import Path

# defaultdict: 대화셋별 발화를 그룹핑할 때, 키가 없어도 자동으로 빈 리스트를 생성하는 딕셔너리
from collections import defaultdict

# load_dotenv: .env 파일의 환경변수를 현재 프로세스에 로드하여
# 데이터 경로를 코드에 직접 노출하지 않고 관리할 수 있게 합니다.
# 이를 통해 .gitignore로 .env를 제외하여 민감한 경로 정보를 보호합니다.
from dotenv import load_dotenv

# .env 파일에서 TRAIN_DATA_DIR, VAL_DATA_DIR 환경변수를 로드합니다.
# 이후 os.getenv()로 데이터 디렉토리 경로를 가져올 수 있습니다.
load_dotenv()

In [ ]:
# =============================================================================
# 2. K쇼핑 콜센터 데이터 로드 (zip -> JSON -> DataFrame)
# =============================================================================
# [의도] 원본 데이터를 읽어서 머신러닝에 사용할 수 있는 DataFrame 형태로 변환합니다.
#
# [핵심 설계 결정 - 대화셋 단위 결합]
#   원본 데이터는 한 통화(대화셋) 안에 여러 개의 발화(turn)가 존재합니다.
#   개별 발화만으로는 맥락이 부족하여 분류 정확도가 떨어지므로,
#   같은 대화셋의 발화들을 하나의 텍스트로 결합합니다.
#
# [v2 개선 - 텍스트 구성 전략]
#   v1에서는 고객 발화만 사용했으나, v2부터 3가지 정보를 결합합니다:
#   1) 고객 질문: 문의의 핵심 내용
#   2) 상담원 응답(최대 3개): "반품 접수 도와드리겠습니다" 같은 응답이
#      카테고리 판별에 직접적인 단서를 제공합니다.
#   3) 고객 의도: "방송상품주문", "상품교환접수" 같은 의도 라벨이
#      분류기에 강력한 힌트를 줍니다.
#
# [v3 개선 - 카테고리 재정의]
#   데이터 분석 결과, "업무처리" 카테고리의 상위 의도가
#   "방송상품주문", "상품주문신청" 등으로 "주문"과 거의 동일했습니다.
#   독립적인 의미가 없는 catch-all 카테고리로서 v2에서 43%라는 낮은 정확도를 기록했고,
#   다른 카테고리와 심하게 혼동되는 원인이었습니다.
#   따라서 v3에서는 업무처리를 제거하여 7개 -> 6개 카테고리로 축소했습니다.
#
# [데이터 경로]
#   .env 파일에서 관리하여 코드에 직접 노출하지 않습니다.
# =============================================================================

# .env 파일에서 Training/Validation 데이터 디렉토리 경로를 가져옵니다.
train_dir = Path(os.getenv("TRAIN_DATA_DIR"))
val_dir = Path(os.getenv("VAL_DATA_DIR"))

# 분류 대상 카테고리 목록 (v3: 업무처리 제거)
# 각 카테고리는 zip 파일명의 일부로도 사용되므로, 파일명 규칙과 일치해야 합니다.
categories = ["AS", "결제", "교환", "반품", "배송", "주문"]

def load_kshopping_data(base_dir, split_name):
    """
    K쇼핑 zip 파일에서 대화셋 단위로 고객+상담원 발화를 결합하여 추출합니다.
    
    [처리 흐름]
    1) 카테고리별 zip 파일을 열어 JSON 데이터를 파싱
    2) '대화셋일련번호'를 기준으로 같은 통화에 속하는 발화들을 그룹핑
    3) 그룹 내에서 고객 질문, 상담원 응답, 고객 의도를 각각 수집
    4) 3가지 정보를 하나의 텍스트로 결합하여 반환
    
    Args:
        base_dir: zip 파일이 위치한 디렉토리 경로
        split_name: "Training" 또는 "Validation" (zip 파일명에 포함됨)
    
    Returns:
        DataFrame: text, category, dialog_id, num_turns 컬럼을 가진 데이터프레임
    """
    all_dialogs = []
    
    for cat in categories:
        # zip 파일명은 "민원(콜센터) 질의응답_K쇼핑_{카테고리}_{분할}.zip" 규칙을 따릅니다.
        zip_name = f"민원(콜센터) 질의응답_K쇼핑_{cat}_{split_name}.zip"
        zip_path = base_dir / zip_name
        
        # 해당 카테고리의 zip 파일이 없으면 건너뜁니다.
        if not zip_path.exists():
            print(f"파일 없음: {zip_name}")
            continue
        
        # zip 파일 내의 JSON 파일을 읽어 파싱합니다.
        # 각 zip에는 하나의 JSON 파일이 포함되어 있으며,
        # JSON은 발화 단위 딕셔너리의 리스트입니다.
        with zipfile.ZipFile(zip_path, 'r') as zf:
            for fname in zf.namelist():
                with zf.open(fname) as f:
                    data = json.load(f)
        
        # [대화셋 그룹핑]
        # '대화셋일련번호'가 같은 발화들은 같은 통화에 속합니다.
        # defaultdict(list)를 사용하여 키가 없는 경우에도 자동으로 빈 리스트가 생성됩니다.
        dialog_groups = defaultdict(list)
        for d in data:
            dialog_groups[d["대화셋일련번호"]].append(d)
        
        # [대화셋별 텍스트 결합]
        # 각 대화셋에서 고객질문 + 상담원응답 + 고객의도를 결합합니다.
        for dialog_id, turns in dialog_groups.items():
            # 고객 질문 수집: QA 필드가 "Q"인 발화에서 고객질문(요청)을 추출합니다.
            customer_texts = [
                d["고객질문(요청)"].strip()
                for d in turns
                if d.get("QA") == "Q" and d.get("고객질문(요청)", "").strip()
            ]
            
            # 상담원 응답 수집: 상담사답변 필드가 비어있지 않은 모든 발화에서 추출합니다.
            # 상담원 응답에는 "반품 접수 도와드리겠습니다" 같은 카테고리 단서가 포함되어 있어
            # 분류 정확도 향상에 기여합니다 (v2에서 추가).
            agent_texts = [
                d["상담사답변"].strip()
                for d in turns
                if d.get("상담사답변", "").strip()
            ]
            
            # 고객 의도 수집: 중복 제거(set)를 적용하여 고유한 의도만 남깁니다.
            # "방송상품주문", "상품교환접수" 같은 라벨로, 분류 단서 역할을 합니다.
            intents = list(set(
                d["고객의도"].strip()
                for d in turns
                if d.get("고객의도", "").strip()
            ))
            
            # 고객 질문이 있는 대화셋만 사용합니다 (질문 없는 대화셋은 의미 없음).
            if customer_texts:
                # 기본: 모든 고객 질문을 공백으로 연결
                combined = " ".join(customer_texts)
                
                # 상담원 응답 추가: [상담] 태그로 구분하여 뒤에 붙입니다.
                # 최대 3개만 사용하는 이유: 너무 많으면 노이즈가 되어 오히려 성능이 떨어질 수 있음
                if agent_texts:
                    agent_summary = " ".join(agent_texts[:3])
                    combined = combined + " [상담] " + agent_summary
                
                # 고객 의도 추가: [의도] 태그로 구분하여 뒤에 붙입니다.
                if intents:
                    combined = combined + " [의도] " + " ".join(intents)
                
                all_dialogs.append({
                    "text": combined,        # 결합된 전체 텍스트
                    "category": cat,         # 소속 카테고리
                    "dialog_id": dialog_id,  # 원본 추적을 위한 대화셋 ID
                    "num_turns": len(customer_texts)  # 고객 발화 수 (대화 길이 참고용)
                })
        
        # 카테고리별 로드 결과를 출력하여 데이터 규모를 확인합니다.
        print(f"  {cat}: {len([d for d in all_dialogs if d['category']==cat])}건 (대화셋 단위)")
    
    return pd.DataFrame(all_dialogs)

# Training 데이터 로드: 모델 학습에 사용됩니다.
print(" Training 데이터 로드 중...")
train_raw_df = load_kshopping_data(train_dir, "Training")
print(f"\n Training 전체 (대화셋 단위): {len(train_raw_df)}건")

# Validation 데이터 로드: 모델 성능 평가(테스트)에 사용됩니다.
# Training과 별도 데이터를 사용하여 모델의 일반화 능력을 측정합니다.
print("\n Validation 데이터 로드 중...")
val_raw_df = load_kshopping_data(val_dir, "Validation")
print(f"\n Validation 전체 (대화셋 단위): {len(val_raw_df)}건")

In [ ]:
# =============================================================================
# 3. 원본 데이터 품질 확인 (전처리 전)
# =============================================================================
# [의도] 전처리를 적용하기 전에 원본 데이터의 품질 문제를 파악합니다.
# 이 분석 결과를 바탕으로 다음 셀(Cell 4)에서 어떤 전처리를 적용할지 결정합니다.
# 확인하는 항목:
#   1) 텍스트 길이 분포: 너무 짧은 텍스트는 분류 단서가 부족
#   2) 10자 미만 텍스트: "네", "아 그래요?" 같은 의미 없는 짧은 응답
#   3) ㅇㅇㅇ 익명화 토큰: 개인정보 마스킹으로 생긴 노이즈
#   4) 빈번한 중복 텍스트: 카테고리 무관하게 반복되는 텍스트는 분류를 방해
#   5) 카테고리별 분포: 데이터 불균형 정도를 확인
# =============================================================================
print("=== 원본 데이터 품질 분석 ===\n")

# [길이 분포 분석]
# 텍스트 길이의 통계치를 확인하여 전반적인 데이터 특성을 파악합니다.
# 평균과 중앙값의 차이가 크면 극단적으로 긴/짧은 텍스트가 있다는 의미입니다.
lengths = train_raw_df["text"].str.len()
print(f"텍스트 길이 통계:")
print(f"  평균: {lengths.mean():.0f}자, 중앙값: {lengths.median():.0f}자")
print(f"  최소: {lengths.min()}자, 최대: {lengths.max()}자")

# [짧은 텍스트 분석]
# 10자 미만의 텍스트는 "네", "예", "여보세요" 같은 필러 표현이 대부분이며,
# 카테고리를 구분하는 데 아무런 단서를 제공하지 않습니다.
short_mask = lengths < 10
print(f"\n10자 미만 텍스트: {short_mask.sum()}건 ({short_mask.mean()*100:.1f}%)")
if short_mask.sum() > 0:
    print("  예시:", train_raw_df[short_mask]["text"].head(5).tolist())

# [익명화 토큰 분석]
# 원본 데이터에서 개인정보(이름, 전화번호 등)가 "ㅇㅇㅇ"으로 마스킹되어 있습니다.
# 이 토큰은 분류와 무관한 노이즈이므로 전처리에서 제거해야 합니다.
anon_mask = train_raw_df["text"].str.contains("ㅇㅇ")
print(f"\nㅇㅇㅇ(익명화) 포함: {anon_mask.sum()}건 ({anon_mask.mean()*100:.1f}%)")

# [빈번한 중복 텍스트 분석]
# 여러 카테고리에 동일하게 등장하는 텍스트는 분류기를 혼란시킵니다.
# 가장 자주 반복되는 텍스트가 무엇인지 확인하여 노이즈 여부를 판단합니다.
from collections import Counter
all_texts = train_raw_df["text"].tolist()
text_counter = Counter(all_texts)
print(f"\n가장 빈번한 텍스트 (노이즈 후보):")
for text, count in text_counter.most_common(5):
    print(f"  [{count}회] {text[:50]}...")

# [카테고리별 분포]
# 카테고리 간 데이터 수의 차이(불균형)를 확인합니다.
# 불균형이 심하면 균등 샘플링이 필요합니다 (Cell 6에서 처리).
print(f"\n=== 카테고리별 분포 ===")
print(train_raw_df["category"].value_counts().sort_index())

---
## Step 2: 데이터 전처리

위 분석에서 발견한 문제점을 해결합니다:

| 문제 | 해결 방법 |
|------|----------|
| 의미 없는 짧은 텍스트 | 최소 글자 수 필터링 (15자 이상) |
| ㅇㅇㅇ 익명화 토큰 | 정규식으로 제거 |
| 불필요한 공백/특수문자 | 정규화 처리 |
| "아 그래요?" 같은 필러 표현 | 필러 패턴 제거 |
| 중복 텍스트 | 중복 제거 (deduplicate) |
| 카테고리 불균형 | 균등 샘플링 |

In [ ]:
# =============================================================================
# 4. 전처리 함수 정의 및 적용
# =============================================================================
# [의도] 앞의 품질 분석(Cell 3)에서 발견한 노이즈를 체계적으로 제거합니다.
# 전처리를 통해 분류에 불필요한 정보를 제거하고, 의미 있는 텍스트만 남깁니다.
#
# [중요] clean_text() 함수는 학습 데이터뿐 아니라 예측 시에도 동일하게 적용됩니다.
# 학습 데이터와 입력 데이터에 같은 전처리를 적용해야 임베딩 공간에서의 유사도가
# 정확하게 계산됩니다. (Cell 11의 predict_inquiry에서 재사용)
# =============================================================================

def clean_text(text):
    """
    텍스트 정제: 노이즈 제거 및 정규화를 수행합니다.
    
    [처리 순서와 이유]
    1) 익명화 토큰 제거: ㅇㅇㅇ 같은 마스킹 토큰은 의미가 없음
    2) 필러 표현 제거: 카테고리 분류에 도움이 되지 않는 반응형 표현
    3) 연속 공백 정리: 삭제로 인해 생긴 불필요한 공백 통합
    4) 양쪽 구두점 정리: 문장 앞뒤의 불필요한 기호 제거
    
    Args:
        text: 정제 대상 원본 텍스트
    Returns:
        정제된 텍스트 문자열
    """
    # 1) ㅇㅇㅇ 익명화 토큰 제거
    # 원본 데이터에서 개인정보(이름, 주소 등)가 "ㅇ"의 연속으로 마스킹되어 있습니다.
    # 'ㅇ'이 2개 이상 연속되는 패턴을 빈 문자열로 대체합니다.
    text = re.sub(r'ㅇ{2,}', '', text)
    
    # 2) 필러(filler) 표현 제거
    # 콜센터 대화에서 자주 등장하지만 분류에 도움이 되지 않는 반응형 표현들입니다.
    # 이러한 표현은 모든 카테고리에 공통으로 등장하므로, 남겨두면 분류기가
    # 카테고리 간 차이를 구분하기 어려워집니다.
    filler_patterns = [
        r'\b아\s*그래요\??',       # "아 그래요?" - 단순 리액션
        r'\b네[\.\s]*$',           # "네." / "네" - 긍정 응답
        r'\b예[\.\s]*$',           # "예." - 긍정 응답
        r'\b여보세요\??',          # "여보세요?" - 통화 시작 인사
        r'\b아\s*네[\.\s]*',       # "아 네." - 리액션
        r'\b잠깐만요[\.\s]*',      # "잠깐만요." - 대기 요청
        r'\b감사합니다[\.\s]*',    # "감사합니다." - 인사
        r'\b수고하십니다[\.\s]*',  # "수고하십니다." - 인사
        r'\b안녕하세요[\.\?\s]*',  # "안녕하세요?" - 통화 시작 인사
    ]
    for pattern in filler_patterns:
        text = re.sub(pattern, ' ', text)
    
    # 3) 연속 공백 정리
    # 위의 삭제 작업으로 인해 공백이 여러 개 연속될 수 있으므로 하나로 통합합니다.
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 4) 양쪽 구두점 정리
    # 문장 앞뒤에 남아있는 마침표, 쉼표, 느낌표 등의 불필요한 기호를 제거합니다.
    text = text.strip('.,!? ')
    
    return text

def preprocess_dataframe(df, min_length=15):
    """
    DataFrame 전체에 전처리 파이프라인을 적용합니다.
    
    [파이프라인 단계]
    1) clean_text 적용: 각 텍스트에 노이즈 제거 수행
    2) 빈 텍스트 제거: 정제 후 내용이 없어진 텍스트 삭제
    3) 최소 길이 필터링: 15자 미만은 분류 단서가 부족하므로 제거
       (15자 기준은 "교환해주세요", "반품하고 싶어요" 같은 최소 의미 단위를 기준으로 설정)
    4) 완전 중복 제거: 동일한 텍스트가 여러 건 있으면 하나만 남김
    
    [각 단계별 제거 건수를 출력하여 전처리 효과를 확인할 수 있습니다]
    
    Args:
        df: 전처리 대상 DataFrame (text, category 컬럼 필수)
        min_length: 최소 텍스트 길이 (기본값 15자)
    Returns:
        전처리 완료된 DataFrame (인덱스 리셋됨)
    """
    original_count = len(df)
    
    # Step 1: 텍스트 정제 - clean_text 함수를 모든 행에 적용합니다.
    df = df.copy()  # 원본 DataFrame을 변경하지 않기 위해 복사합니다.
    df["text_clean"] = df["text"].apply(clean_text)
    
    # Step 2: 빈 텍스트 제거 - 정제 후 내용이 모두 사라진 텍스트를 삭제합니다.
    df = df[df["text_clean"].str.len() > 0]
    after_empty = len(df)
    
    # Step 3: 최소 길이 필터링
    # 15자 미만의 텍스트는 "네 알겠습니다", "확인해주세요" 같이
    # 어떤 카테고리에도 해당할 수 있는 범용적 표현이 대부분입니다.
    # 이런 텍스트를 남겨두면 분류기가 혼동하기 때문에 제거합니다.
    df = df[df["text_clean"].str.len() >= min_length]
    after_length = len(df)
    
    # Step 4: 완전 중복 제거
    # 동일한 텍스트가 반복되면 학습 데이터에 편향이 생기므로 하나만 남깁니다.
    df = df.drop_duplicates(subset=["text_clean"])
    after_dedup = len(df)
    
    # 정제된 텍스트로 원본을 교체하고 임시 컬럼을 삭제합니다.
    df["text"] = df["text_clean"]
    df = df.drop(columns=["text_clean"])
    
    # 각 단계별 제거 건수를 출력하여 전처리의 효과를 정량적으로 확인합니다.
    print(f"  원본: {original_count}건")
    print(f"  -> 빈 텍스트 제거 후: {after_empty}건 (-{original_count - after_empty})")
    print(f"  -> {min_length}자 미만 제거 후: {after_length}건 (-{after_empty - after_length})")
    print(f"  -> 중복 제거 후: {after_dedup}건 (-{after_length - after_dedup})")
    
    return df.reset_index(drop=True)

# 전처리 적용
# Training 데이터와 Validation 데이터에 동일한 전처리를 적용합니다.
# 학습과 평가에 동일한 기준을 적용해야 공정한 성능 측정이 가능합니다.
print("Training 데이터 전처리 중...")
train_clean_df = preprocess_dataframe(train_raw_df)

print("\nValidation 데이터 전처리 중...")
val_clean_df = preprocess_dataframe(val_raw_df)

In [ ]:
# =============================================================================
# 5. 전처리 전 vs 후 비교
# =============================================================================
# [의도] 전처리가 실제로 데이터 품질을 개선했는지 눈으로 확인합니다.
# 전처리 전후의 텍스트 예시와 길이 통계를 비교하여,
# 노이즈가 제대로 제거되었는지, 의미 있는 내용이 손상되지 않았는지 검증합니다.
# 이 단계는 전처리 파이프라인의 효과를 정성적/정량적으로 확인하는 역할을 합니다.
# =============================================================================
print("=== 전처리 전 vs 후 비교 ===\n")

# [전처리 전 예시]
# 원본 텍스트에 포함된 ㅇㅇㅇ, 필러 표현 등 노이즈를 직접 확인할 수 있습니다.
print(" 전처리 전 (원본):")
for i, row in train_raw_df.head(3).iterrows():
    print(f"  [{row['category']}] {row['text'][:80]}...")

# [전처리 후 예시]
# 노이즈가 제거된 깔끔한 텍스트를 확인합니다.
# 원본과 비교하여 핵심 내용은 유지되면서 불필요한 부분만 제거되었는지 확인합니다.
print(f"\n 전처리 후 (정제):")
for i, row in train_clean_df.head(3).iterrows():
    print(f"  [{row['category']}] {row['text'][:80]}...")

# [길이 분포 비교]
# 전처리 후 평균/중앙값 길이가 어떻게 변했는지 확인합니다.
# 노이즈 제거로 인해 길이가 약간 줄어들 수 있지만,
# 핵심 내용이 유지되므로 큰 차이가 나지 않아야 합니다.
print(f"\n 텍스트 길이 비교:")
print(f"  전처리 전 -- 평균: {train_raw_df['text'].str.len().mean():.0f}자, 중앙값: {train_raw_df['text'].str.len().median():.0f}자")
print(f"  전처리 후 -- 평균: {train_clean_df['text'].str.len().mean():.0f}자, 중앙값: {train_clean_df['text'].str.len().median():.0f}자")

# [전처리 후 카테고리 분포]
# 전처리 과정에서 특정 카테고리의 데이터가 과도하게 제거되지 않았는지 확인합니다.
# 카테고리 간 비율이 크게 변했다면 전처리 기준을 재검토해야 합니다.
print(f"\n 전처리 후 카테고리 분포:")
print(train_clean_df["category"].value_counts().sort_index())

In [ ]:
# =============================================================================
# 6. 균등 샘플링 (카테고리 균형 맞추기)
# =============================================================================
# [의도] 카테고리별 데이터 수가 다르면 분류기가 데이터가 많은 카테고리 쪽으로
# 편향되어 학습됩니다. 이를 방지하기 위해 모든 카테고리에서 동일한 수의
# 샘플을 추출하여 균형 잡힌 학습/테스트 데이터를 만듭니다.
#
# [v2 개선] 샘플 수를 v1에서 대폭 증가시켰습니다.
#   - Training: 500건 -> 1000건 (더 많은 패턴을 학습)
#   - Test: 100건 -> 200건 (더 안정적인 성능 측정)
#   데이터가 충분할 때 샘플 수를 늘리면 분류기가 더 다양한 표현을 학습하여
#   일반화 능력이 향상됩니다.
#
# [min() 사용 이유] 특정 카테고리의 데이터가 목표 수보다 적을 수 있으므로,
# 실제 데이터 수와 목표 수 중 작은 값을 사용합니다.
# =============================================================================
SAMPLE_PER_CAT_TRAIN = 1000  # 카테고리당 학습 샘플 수 (v1: 500 -> v2: 1000)
SAMPLE_PER_CAT_TEST = 200    # 카테고리당 테스트 샘플 수 (v1: 100 -> v2: 200)

# [Training 데이터 균등 샘플링]
# 각 카테고리에서 동일한 수의 샘플을 무작위 추출합니다.
# random_state=42로 재현성을 보장합니다 (같은 코드 실행 시 동일한 결과).
train_samples = []
for cat in categories:
    subset = train_clean_df[train_clean_df["category"] == cat]
    n = min(len(subset), SAMPLE_PER_CAT_TRAIN)
    train_samples.append(subset.sample(n=n, random_state=42))

# 카테고리별 샘플을 하나의 DataFrame으로 합칩니다.
# reset_index(drop=True)로 인덱스를 0부터 재정렬합니다.
train_df = pd.concat(train_samples).reset_index(drop=True)

# [Test 데이터 균등 샘플링]
# Validation 데이터에서 동일한 방식으로 테스트 세트를 구성합니다.
# Training과 별도의 데이터를 사용하여 모델의 일반화 능력을 측정합니다.
test_samples = []
for cat in categories:
    subset = val_clean_df[val_clean_df["category"] == cat]
    n = min(len(subset), SAMPLE_PER_CAT_TEST)
    test_samples.append(subset.sample(n=n, random_state=42))
test_df = pd.concat(test_samples).reset_index(drop=True)

# [결과 출력] 최종 데이터셋의 크기와 카테고리별 분포를 확인합니다.
# 모든 카테고리의 건수가 동일해야 올바르게 균등 샘플링된 것입니다.
print(f"최종 train: {len(train_df)}건")
print(f"최종 test: {len(test_df)}건")
print(f"\n=== Train 카테고리별 분포 ===")
print(train_df["category"].value_counts().sort_index())
print(f"\n=== Test 카테고리별 분포 ===")
print(test_df["category"].value_counts().sort_index())

# [샘플 확인] 각 카테고리의 대표 텍스트를 출력하여 데이터 품질을 확인합니다.
# 전처리가 올바르게 적용된 깔끔한 텍스트인지 눈으로 검증합니다.
print("\n=== 전처리된 질문 예시 ===")
for cat in categories:
    sample = train_df[train_df["category"] == cat]["text"].iloc[0]
    print(f"\n [{cat}]")
    print(f"   -> {sample[:120]}{'...' if len(sample)>120 else ''}")

In [ ]:
# =============================================================================
# 7. 라벨 인코딩 (카테고리 문자열 -> 숫자 변환)
# =============================================================================
# [의도] 머신러닝 분류기는 숫자만 처리할 수 있으므로,
# 문자열 카테고리("AS", "결제", "교환" 등)를 정수(0, 1, 2 등)로 변환합니다.
#
# [LabelEncoder의 역할]
#   - fit_transform(): Training 데이터의 카테고리 목록을 학습하고 동시에 변환합니다.
#     이때 카테고리와 숫자 간의 매핑 테이블이 내부에 저장됩니다.
#   - transform(): 저장된 매핑 테이블을 사용하여 Test 데이터를 변환합니다.
#     동일한 매핑을 사용해야 Train과 Test의 라벨이 일관성을 유지합니다.
#
# [예측 시 역변환]
#   분류기의 예측 결과(숫자)를 다시 카테고리명(문자열)으로 변환할 때
#   label_encoder.classes_ 배열을 사용합니다 (Cell 11 predict_inquiry 참조).
# =============================================================================
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Training 데이터: 카테고리 목록 학습(fit) + 숫자 변환(transform)을 동시에 수행합니다.
train_df["label"] = label_encoder.fit_transform(train_df["category"])

# Test 데이터: Training에서 학습한 동일한 매핑으로 변환합니다.
test_df["label"]  = label_encoder.transform(test_df["category"])

# 매핑 결과 출력: 어떤 카테고리가 어떤 숫자에 대응되는지 확인합니다.
print("카테고리 -> 숫자 매핑:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i}: {cls}")

---
## Step 3: 임베딩 & 분류기 학습

In [ ]:
# =============================================================================
# 8. 문장 임베딩 (텍스트 -> 고차원 벡터 변환)
# =============================================================================
# [의도] 텍스트를 분류기가 처리할 수 있는 수치 벡터(임베딩)로 변환합니다.
# 임베딩 모델은 의미가 비슷한 문장을 벡터 공간에서 가까운 위치에 배치하므로,
# 분류기가 카테고리별 패턴을 학습할 수 있게 됩니다.
#
# [v3 개선 - 모델 교체]
#   v1-v2: MiniLM-L12 (384차원) - 경량 모델, 빠르지만 의미 포착 능력이 제한적
#   v3:    MPNet-base (768차원) - 더 큰 모델, 2배 넓은 벡터 공간에서
#          미묘한 의미 차이를 더 정밀하게 구분합니다.
#          예: "교환"과 "반품"처럼 비슷하지만 다른 개념의 차이를 더 잘 포착합니다.
#
# [paraphrase-multilingual 시리즈를 선택한 이유]
#   한국어를 포함한 50개 이상의 언어를 지원하는 다국어 모델이며,
#   의미가 같은 문장(paraphrase)을 유사한 벡터로 변환하도록 사전학습되었습니다.
#
# [batch_size=64의 의미]
#   한 번에 64개의 텍스트를 묶어서 변환합니다.
#   배치 크기가 클수록 빠르지만 메모리를 더 많이 사용합니다.
# =============================================================================
from sentence_transformers import SentenceTransformer

# 사전학습된 다국어 MPNet 모델을 로드합니다.
# 최초 실행 시 모델 파일(약 1GB)이 자동으로 다운로드됩니다.
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

def embed_texts(texts, batch_size=64):
    """
    텍스트 리스트를 임베딩 벡터 배열로 변환합니다.
    
    Args:
        texts: 변환할 텍스트 리스트
        batch_size: 한 번에 처리할 텍스트 수 (기본값 64)
    Returns:
        numpy 배열 (shape: [텍스트 수, 768])
    """
    return model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

# Training 데이터 임베딩: 분류기 학습에 사용할 벡터를 생성합니다.
print(" train 임베딩 중... (768차원 MPNet 모델)")
X_train = embed_texts(train_df["text"].tolist())
y_train = train_df["label"].values  # 대응하는 라벨 (정수)

# Test 데이터 임베딩: 성능 평가에 사용할 벡터를 생성합니다.
print(" test 임베딩 중...")
X_test = embed_texts(test_df["text"].tolist())
y_test = test_df["label"].values

# [shape 확인] (데이터 수, 768) 형태여야 정상입니다.
# 768은 MPNet 모델의 출력 차원으로, 각 텍스트가 768개의 숫자로 표현됩니다.
print(f"\n임베딩 shape: train {X_train.shape}, test {X_test.shape}")

In [ ]:
# =============================================================================
# 9. 분류기 비교 학습 (로지스틱 회귀 vs SVM)
# =============================================================================
# [의도] 두 가지 전통 머신러닝 분류기를 학습시키고, 성능이 더 좋은 모델을
# 자동으로 선택합니다. 서로 다른 학습 방식을 비교함으로써 데이터 특성에
# 가장 적합한 분류기를 찾습니다.
#
# [v2에서 추가] v1에서는 로지스틱 회귀만 사용했으나,
# v2부터 SVM을 추가하여 두 모델을 비교합니다.
#
# [두 분류기의 차이]
#   - 로지스틱 회귀: 선형 결정 경계를 사용. 단순하고 빠르지만,
#     카테고리 간 비선형적 관계는 포착하기 어렵습니다.
#   - SVM (RBF 커널): 비선형 결정 경계를 사용. 고차원 임베딩 벡터에서
#     복잡한 패턴을 감지할 수 있어 더 정확한 분류가 가능합니다.
#
# [clf.fit()이 학습 단계]
#   fit() 호출 시 분류기가 X_train(임베딩 벡터)과 y_train(라벨) 사이의
#   관계를 학습합니다. 이것이 이 파이프라인에서 "훈련"에 해당합니다.
#   (임베딩 모델 자체는 사전학습된 것을 그대로 사용하고, 여기서 fine-tuning하지 않습니다.)
# =============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# [모델 1: 로지스틱 회귀]
# max_iter=2000: 수렴할 때까지 최대 2000번 반복합니다.
# 기본값(100)으로는 768차원 고차원 데이터에서 수렴하지 않을 수 있어 늘렸습니다.
clf_lr = LogisticRegression(max_iter=2000)
print(" [1/2] 로지스틱 회귀 학습 중...")
clf_lr.fit(X_train, y_train)              # 학습 (임베딩 -> 라벨 매핑 학습)
y_pred_lr = clf_lr.predict(X_test)        # 테스트 데이터에 대한 예측
acc_lr = accuracy_score(y_test, y_pred_lr)  # 정확도 계산 (맞춘 비율)
print(f"   -> Accuracy: {acc_lr:.4f}")

# [모델 2: SVM (RBF 커널)]
# kernel='rbf': 방사 기저 함수 커널. 비선형 결정 경계를 만들어
#   임베딩 벡터 공간에서 카테고리별 영역을 유연하게 구분합니다.
# C=10: 규제 파라미터. 값이 클수록 학습 데이터에 대한 정확도를 높이려 하지만,
#   너무 크면 과적합(overfitting)될 위험이 있습니다. 10은 적절한 균형점입니다.
# gamma='scale': 커널 함수의 영향 범위를 자동으로 설정합니다.
#   1 / (n_features * X.var()) 공식으로 계산됩니다.
# probability=True: predict_proba() 메서드를 사용하기 위해 필요합니다.
#   예측 시 각 카테고리별 확률을 출력할 수 있게 됩니다 (Cell 11에서 사용).
clf_svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True)
print(" [2/2] SVM (RBF) 학습 중...")
clf_svm.fit(X_train, y_train)
y_pred_svm = clf_svm.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)
print(f"   -> Accuracy: {acc_svm:.4f}")

# [자동 모델 선택]
# 테스트 정확도가 더 높은 모델을 최종 분류기(clf)로 선택합니다.
# 이후 예측(Cell 11)과 혼동 행렬(Cell 10)에서 이 clf를 사용합니다.
if acc_svm >= acc_lr:
    clf = clf_svm
    y_pred = y_pred_svm
    acc = acc_svm
    best_name = "SVM (RBF)"
else:
    clf = clf_lr
    y_pred = y_pred_lr
    acc = acc_lr
    best_name = "로지스틱 회귀"

print(f"\n 최종 선택: {best_name} (Accuracy: {acc:.4f})")

# [분류 리포트 출력]
# precision: 해당 카테고리로 예측한 것 중 실제로 맞은 비율
# recall: 실제 해당 카테고리인 것 중 올바르게 예측한 비율
# f1-score: precision과 recall의 조화 평균 (종합 성능 지표)
# support: 해당 카테고리의 실제 테스트 데이터 수
print(f"\n=== 분류 리포트 ({best_name}) ===")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# =============================================================================
# 10. 혼동 행렬 (Confusion Matrix) 시각화
# =============================================================================
# [의도] 분류기가 어떤 카테고리를 잘 맞추고, 어떤 카테고리를 헷갈려하는지
# 시각적으로 파악합니다.
#
# [혼동 행렬 읽는 방법]
#   - 행(가로) = 실제 카테고리, 열(세로) = 예측 카테고리
#   - 대각선(좌상단->우하단): 올바르게 분류한 건수 (높을수록 좋음)
#   - 대각선 외 셀: 잘못 분류한 건수 (어떤 카테고리와 혼동하는지 파악 가능)
#   - 예: (교환 행, 반품 열)에 숫자가 크면, 교환을 반품으로 잘못 분류하는 경우가 많다는 의미
# =============================================================================
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import matplotlib

# [한글 폰트 설정]
# matplotlib의 기본 폰트는 한글을 지원하지 않으므로,
# macOS의 AppleGothic 폰트를 지정하여 한글 깨짐을 방지합니다.
# unicode_minus=False로 마이너스 기호가 깨지는 문제도 해결합니다.
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# [혼동 행렬 계산]
# y_test(실제 라벨)와 y_pred(예측 라벨)을 비교하여
# 각 (실제, 예측) 조합의 건수를 세어 행렬로 만듭니다.
cm = confusion_matrix(y_test, y_pred)

# [히트맵 시각화]
# imshow()로 행렬 값을 색상 강도로 표현합니다.
# Blues 컬러맵을 사용하여 값이 클수록 진한 파란색으로 표시됩니다.
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)  # 오른쪽에 색상 범례를 추가합니다.

classes = label_encoder.classes_
ax.set(xticks=range(len(classes)), yticks=range(len(classes)),
       xticklabels=classes, yticklabels=classes,
       ylabel='실제 카테고리', xlabel='예측 카테고리',
       title=f'혼동 행렬 ({best_name}, Accuracy: {acc:.4f})')

# [셀 안에 숫자 표시]
# 각 셀에 건수를 직접 표시하여 정확한 값을 확인할 수 있게 합니다.
# thresh를 기준으로 배경이 어두우면 흰색, 밝으면 검은색 글씨를 사용하여
# 가독성을 확보합니다.
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black", fontsize=11)

plt.tight_layout()
plt.show()

# [카테고리별 정확도 요약]
# 혼동 행렬의 대각선 값을 해당 행의 합계로 나누어 카테고리별 정확도를 계산합니다.
# 이를 통해 어떤 카테고리의 분류가 특히 어려운지 한눈에 파악할 수 있습니다.
print("\n 카테고리별 정확도:")
for i, cls in enumerate(classes):
    correct = cm[i, i]      # 대각선: 올바르게 분류한 건수
    total = cm[i].sum()      # 해당 행 전체: 실제 해당 카테고리의 총 건수
    print(f"  {cls:8s}: {correct}/{total} ({correct/total*100:.1f}%)")

---
## Step 4: 예측 & 평가

In [ ]:
# =============================================================================
# 11. 예측 함수 정의 (새로운 문의를 분류)
# =============================================================================
# [의도] 학습이 완료된 분류기를 사용하여 새로운 고객 문의 텍스트의
# 카테고리를 예측하는 함수를 정의합니다.
#
# [핵심 포인트 - 전처리 일관성]
#   입력 텍스트에도 학습 데이터와 동일한 clean_text() 전처리를 적용합니다.
#   학습 시 전처리된 텍스트로 임베딩을 생성했으므로,
#   예측 시에도 같은 전처리를 거쳐야 임베딩 공간에서 올바른 위치에 매핑됩니다.
#   전처리를 생략하면 노이즈로 인해 잘못된 벡터가 생성되어 예측 정확도가 떨어집니다.
#
# [predict_proba의 역할]
#   일반 predict()는 가장 확률이 높은 카테고리 하나만 반환하지만,
#   predict_proba()는 모든 카테고리에 대한 확률 분포를 반환합니다.
#   이를 통해 분류기의 "확신 정도"를 파악할 수 있습니다.
# =============================================================================
def predict_inquiry(text: str, top_k: int = 3):
    """
    고객 문의 텍스트를 입력받아 유형을 예측합니다.
    
    [처리 흐름]
    1) clean_text()로 입력 텍스트 전처리
    2) 임베딩 모델로 벡터 변환
    3) 분류기로 각 카테고리별 확률 계산
    4) 상위 top_k 카테고리를 확률과 함께 막대 그래프로 출력
    
    Args:
        text: 분류할 고객 문의 텍스트 (원본 그대로 입력)
        top_k: 출력할 상위 카테고리 수 (기본값 3)
    """
    # 입력 텍스트에 학습 데이터와 동일한 전처리를 적용합니다.
    cleaned = clean_text(text)
    
    # 전처리된 텍스트를 768차원 벡터로 변환합니다.
    # [cleaned]로 리스트 형태로 전달해야 model.encode가 정상 동작합니다.
    emb = model.encode([cleaned], convert_to_numpy=True)
    
    # 분류기를 통해 각 카테고리에 속할 확률을 계산합니다.
    # probs는 [p_AS, p_결제, p_교환, p_반품, p_배송, p_주문] 형태의 배열입니다.
    probs = clf.predict_proba(emb)[0]
    
    # 확률이 높은 순서대로 top_k개의 인덱스를 추출합니다.
    # np.argsort(-probs)는 확률을 내림차순으로 정렬한 인덱스를 반환합니다.
    top_indices = np.argsort(-probs)[:top_k]

    # [결과 출력]
    print("\n" + "=" * 50)
    print(" 고객 문의 내용:")
    print(f"   {text}")
    # 전처리로 텍스트가 변경된 경우, 변경된 내용도 함께 보여줍니다.
    if cleaned != text:
        print(f"   (전처리 후: {cleaned[:60]}{'...' if len(cleaned)>60 else ''})")
    print("=" * 50)
    
    # 상위 카테고리를 확률과 함께 시각적 막대 그래프로 출력합니다.
    # 블록 문자(block)의 길이로 확률의 상대적 크기를 직관적으로 보여줍니다.
    print("\n 예측된 문의 유형 (Top-k):")
    for idx in top_indices:
        cat_name = label_encoder.classes_[idx]  # 숫자 인덱스를 카테고리명으로 역변환
        p = probs[idx]
        bar = "█" * int(p * 20)  # 확률에 비례하는 길이의 막대 생성 (최대 20칸)
        print(f"   {cat_name:8s} {bar} ({p:.3f})")
    print("=" * 50 + "\n")

In [ ]:
# =============================================================================
# 테스트: 명확한 문의 3건으로 분류기 동작 확인
# =============================================================================
# [의도] 각 카테고리에 명확하게 해당하는 문의를 입력하여
# 분류기가 올바르게 작동하는지 기본적인 검증을 수행합니다.
#
# [테스트 케이스 선정 기준]
# 사람이 보기에도 카테고리가 명확한 문장을 선택하여,
# 분류기의 기본적인 판별 능력을 먼저 확인합니다.
# 이후 Mission에서 애매한 경우도 테스트합니다.
# =============================================================================

# 테스트 1: "배송 조회" 키워드가 포함된 명확한 배송 문의
print(">>> 테스트 1: 배송 문의")
predict_inquiry("주문한 물건이 아직 안 왔어요. 배송 조회 좀 해주세요.")

# 테스트 2: "환불", "반품 접수" 키워드가 포함된 명확한 반품 문의
print(">>> 테스트 2: 반품 문의")
predict_inquiry("이 제품 환불하고 싶어요. 반품 접수해주세요.")

# 테스트 3: "카드 결제", "결제 수단" 키워드가 포함된 명확한 결제 문의
print(">>> 테스트 3: 결제 문의")
predict_inquiry("카드 결제가 안 돼요. 다른 결제 수단으로 바꿀 수 있나요?")

---
## Step 5: 미니 미션

### 미션 1: 직접 고객 문의 작성하기
각 카테고리에 맞는 고객 문의를 직접 작성하고, AI가 올바르게 분류하는지 확인하세요.

In [ ]:
# =============================================================================
# 미션 1-1: AS 관련 문의
# =============================================================================
# [의도] AS(After Service) 카테고리에 해당하는 문의를 직접 작성하여
# 분류기가 올바르게 판별하는지 확인합니다.
# AS 카테고리는 제품 고장, 수리, 불량 등 물리적 하자와 관련된 문의입니다.
# =============================================================================
my_inquiry_1 = "TV 화면이 깨져서 나와요. AS 접수하려면 어떻게 해야 하나요?"
predict_inquiry(my_inquiry_1)
# 예상한 카테고리: AS
# AI의 예측이 맞았나요?: 

In [ ]:
# =============================================================================
# 미션 1-2: 교환 관련 문의
# =============================================================================
# [의도] 교환 카테고리에 해당하는 문의로 분류기를 테스트합니다.
# 교환은 동일 상품의 다른 옵션(사이즈, 색상 등)으로 바꾸는 것을 의미합니다.
# 반품(환불)과 혼동하기 쉬운 카테고리이므로 분류기의 구분 능력을 확인합니다.
# =============================================================================
my_inquiry_2 = "사이즈가 안 맞아서 교환하고 싶어요. 다른 사이즈로 바꿀 수 있나요?"
predict_inquiry(my_inquiry_2)
# 예상한 카테고리: 교환
# AI의 예측이 맞았나요?: 

In [ ]:
# =============================================================================
# 미션 1-3: 주문 관련 문의
# =============================================================================
# [의도] 주문 카테고리에 해당하는 문의로 분류기를 테스트합니다.
# 주문은 상품 주문, 주문 취소, 주문 내역 확인 등과 관련된 문의입니다.
# v3에서 업무처리 카테고리를 제거하면서 주문 분류 정확도가 크게 개선되었습니다.
# =============================================================================
my_inquiry_3 = "주문을 취소하고 싶은데요. 아직 출발 전이면 취소되나요?"
predict_inquiry(my_inquiry_3)
# 예상한 카테고리: 주문
# AI의 예측이 맞았나요?: 

### 미션 2: 경계가 애매한 문의 만들기
하나의 문의가 **두 가지 카테고리에 동시에 해당**할 수 있는 문장을 만들어보세요.  
예: "주문한 물건이 불량이라 교환하고 싶어요" → 주문? 교환? AS?

In [ ]:
# =============================================================================
# 미션 2-1: 교환 + 반품이 애매한 문의
# =============================================================================
# [의도] 두 카테고리에 동시에 해당할 수 있는 애매한 문장을 입력하여
# 분류기가 어떻게 판단하는지, 그리고 확률 분포가 어떻게 나오는지 확인합니다.
# "바꾸거나"(교환) + "돌려보내고"(반품)이 함께 등장하는 경계 사례입니다.
# 이런 경우 top_k 확률에서 두 카테고리가 비슷한 확률로 나타날 수 있습니다.
# =============================================================================
ambiguous_1 = "상품이 마음에 안 들어서 다른 걸로 바꾸거나 그냥 돌려보내고 싶어요."
predict_inquiry(ambiguous_1)
# 해당할 수 있는 카테고리 2개: 
# AI 예측이 적절한가요?: 

In [ ]:
# =============================================================================
# 미션 2-2: 결제 + 주문이 애매한 문의
# =============================================================================
# [의도] "주문하면서 결제"라는 표현이 주문과 결제 두 카테고리에 모두 걸립니다.
# 분류기가 맥락을 종합적으로 판단하여 더 적절한 카테고리를 선택하는지 확인합니다.
# "금액이 이상해요"는 결제 쪽에 가까운 단서이므로, 결제로 분류되면 적절합니다.
# =============================================================================
ambiguous_2 = "주문하면서 결제했는데 금액이 이상해요. 확인 좀 해주세요."
predict_inquiry(ambiguous_2)
# 해당할 수 있는 카테고리 2개: 
# AI 예측이 적절한가요?: 

In [ ]:
# =============================================================================
# 미션 2-3: 배송 + 반품이 애매한 문의
# =============================================================================
# [의도] "배송 온 물건이 파손"(배송 문제 + 반품 사유)이 결합된 경계 사례입니다.
# 배송 과정의 문제를 설명하지만, 최종 요청은 반품이므로 복합적 성격을 가집니다.
# 분류기가 전체 맥락에서 핵심 의도를 파악할 수 있는지 테스트합니다.
# =============================================================================
ambiguous_3 = "배송 온 물건이 파손되어 있어서 반품하려고요. 택배 다시 보내야 하나요?"
predict_inquiry(ambiguous_3)
# 해당할 수 있는 카테고리 2개: 
# AI 예측이 적절한가요?: 

### 미션 3: AI 확신도 분석
`predict_inquiry`는 상위 3개 카테고리와 확률을 보여줍니다.

- 1위 확률이 0.8 이상 → AI가 **확신**하는 것
- 1위와 2위 확률이 비슷 → AI가 **헷갈리는** 것

이 두 가지 경우를 각각 찾아보세요.

In [ ]:
# =============================================================================
# 미션 3-1: AI가 확신하는 문의 (1위 확률 0.8 이상 목표)
# =============================================================================
# [의도] 1위 카테고리의 확률이 0.8 이상이면, 분류기가 해당 카테고리에
# 높은 확신을 가지고 있다는 의미입니다.
# "배송", "송장번호" 같은 특정 카테고리에만 등장하는 명확한 키워드가 포함된
# 문장일수록 확신도가 높아집니다.
# =============================================================================
certain_inquiry = "배송이 언제 되나요? 송장번호 알려주세요."
predict_inquiry(certain_inquiry)
# 1위 확률: 

In [ ]:
# =============================================================================
# 미션 3-2: AI가 헷갈리는 문의 (1위와 2위 확률 차이 0.1 이하 목표)
# =============================================================================
# [의도] 1위와 2위 카테고리의 확률 차이가 작으면, 분류기가 두 카테고리 사이에서
# 결정을 내리기 어려워하고 있다는 의미입니다.
# "바꿔주실 수 있나요"(교환) + "돈 돌려받을게요"(반품)처럼 두 카테고리의
# 특징이 모두 포함된 문장이 대표적인 예시입니다.
# 이런 경우가 실무에서 발생하면 사람이 최종 판단해야 합니다.
# =============================================================================
uncertain_inquiry = "이 물건 문제가 있는데 바꿔주실 수 있나요? 아니면 그냥 돈 돌려받을게요."
predict_inquiry(uncertain_inquiry)
# 1위와 2위의 확률 차이: 

### 미션 4: 실제 업무 활용 아이디어
이 분류기를 실제 쇼핑몰 고객센터에 적용한다면 어떻게 활용할 수 있을까요?

In [ ]:
# =============================================================================
# 미션 4: 실제 업무 활용 아이디어
# =============================================================================
# [의도] 기술적 구현을 넘어, 이 분류기가 실제 비즈니스 환경에서
# 어떻게 활용될 수 있는지 생각해보는 확장 과제입니다.
# 기술의 한계와 가능성을 함께 고려하여 현실적인 활용 방안을 모색합니다.
# =============================================================================
#
# 1. 이 분류기를 실제 고객센터에 적용하면 어떤 점이 좋을까요?
#    -> 
#
# 2. 분류 정확도를 높이려면 어떤 방법이 있을까요? (2가지 이상)
#    -> 
#
# 3. 6개 카테고리 외에 추가하면 좋을 카테고리가 있다면?
#    -> 
#
# 4. 이 분류기의 한계점은 무엇일까요?
#    -> 